# 🌍 Traducción masiva es→en con Gemini + fallback local (NLLB)

**Estrategia**

1. **Primario**: Gemini Flash-Lite via API (barato, rápido, buen manejo de lenguaje coloquial).
2. **Fallback**: `facebook/nllb-200-distilled-600M` corriendo en la GPU T4 de Colab.
3. **Por chunks**: cada lote se procesa de manera independiente; si se cae Gemini por rate-limit, el chunk sigue entero con NLLB y la próxima tanda reintenta Gemini cuando pase la ventana de cooldown.
4. **Checkpoint incremental**: cada traducción se persiste como una línea JSONL (`doc_id`, `translation`, `model`, `status`, `ts`). Si el notebook se interrumpe o muere Colab, al re-ejecutar **se salta lo ya hecho** y sigue desde donde quedó.
5. **Resume idempotente**: al arrancar, se reconstruye el set de `doc_id`s ya escritos en el JSONL y nunca se vuelve a traducir uno presente en ese set (los errores previos se respetan salvo que actives `RETRY_ERRORS=True`).

> Salida final: parquet original **+** columna `translation` joinneada por `doc_id`.


In [ ]:
!pip install -q -U requests pandas pyarrow 2>&1 | tail -3

In [ ]:
import os, json, time, sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


# Raíz del proyecto (asume que el notebook vive en notebooks/06_translate/)
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent if "__file__" in dir() \
               else Path(os.getcwd()).resolve()
# Si seguimos sin estar en la raíz, buscar hacia arriba hasta encontrar Makefile
while not (PROJECT_ROOT / "Makefile").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f"CWD fijado a: {PROJECT_ROOT}")


# 1) LM Studio: servidor local compatible con OpenAI (http://localhost:1234)
LMSTUDIO_BASE_URL = "http://localhost:1234/v1"
LMSTUDIO_API_KEY  = "lm-studio"   # LM Studio ignora el valor, pero el cliente lo exige
LMSTUDIO_MODEL    = "qwen2.5-coder-1.5b-instruct"  # @param ["qwen2.5-coder-1.5b-instruct"]

# 2) Idiomas (en -> es; las traducciones existentes están en español)
SRC_LANG = "English"
TGT_LANG = "Spanish"

# 3) Columnas del parquet
TEXT_COL  = "text_clean"
ID_COL    = "doc_id"

# 4) Tamaños de lote
CHUNK_SIZE   = 50   # cuántos textos por *chunk lógico* (se persisten juntos)
BATCH_SIZE   = 4    # cuántos textos por *llamada* a LM Studio dentro del chunk
MAX_RETRIES  = 3     # reintentos por batch ante error de red / formato
BACKOFF_BASE = 2.0   # segundos; se duplica en cada reintento

# 5) Rutas (Windows local, absolutas respecto al PROJECT_ROOT)
BASE_DIR     = PROJECT_ROOT / "notebooks" / "06_translate"
PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_v1.parquet"
JSONL_PATH   = BASE_DIR / "translations.jsonl"
PARQUET_OUT  = BASE_DIR / "translated.parquet"

# 6) Comportamiento
RETRY_ERRORS = False  # si True, también re-traduce los que dieron 'error' la vez pasada

print("Config cargada")
print(f"   LM Studio = {LMSTUDIO_BASE_URL}  ({LMSTUDIO_MODEL})")
print(f"   {SRC_LANG}->{TGT_LANG}  |  chunk={CHUNK_SIZE}  batch={BATCH_SIZE}")
print(f"   parquet in  = {PARQUET_PATH}")
print(f"   jsonl out   = {JSONL_PATH}")

In [6]:
import json, os
from pathlib import Path

def load_done_ids(path: Path) -> set:
    """Lee el JSONL existente y devuelve un set con los doc_id ya procesados.
    Separa 'ok' y 'error' para soportar RETRY_ERRORS."""
    done_ok, done_err = set(), set()
    if not path.exists():
        return done_ok, done_err
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            did = rec.get('doc_id')
            if not did:
                continue
            if rec.get('status') == 'ok':
                done_ok.add(did)
            else:
                done_err.add(did)
    return done_ok, done_err


def append_translations(records, path: Path):
    """Append seguro: una línea JSONL por record. Flush + fsync."""
    if not records:
        return
    with open(path, 'a', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
        f.flush()
        os.fsync(f.fileno())


print("Checkpoint helpers listos")

# Carga del parquet
assert PARQUET_PATH.exists(), f"No existe {PARQUET_PATH}. Ajustá PARQUET_PATH en la celda de config."
df = pd.read_parquet(PARQUET_PATH)
print(f"Parquet cargado: {len(df):,} filas  | columnas = {list(df.columns)}")

Checkpoint helpers listos
Parquet cargado: 1,047,194 filas  | columnas = ['doc_id', 'user_id', 'source', 'timestamp', 'text_raw', 'text_clean', 'label', 'label_source', 'lang', 'corpus_source', 'is_translated']


In [ ]:
import re
import requests

_FENCE_RE = re.compile(r"```(?:json)?", re.IGNORECASE)


def _extract_json_array(content: str):
    """Extrae el primer array JSON válido de un string.

    Robusto a: markdown code fences, texto antes/después del array,
    comillas tipográficas, comillas sin escapar y truncado por max_tokens.
    Usa json.JSONDecoder.raw_decode que parsea desde una posición y devuelve
    el offset de fin — ideal para extraer el primer array completo."""
    if not content:
        return None
    s = content.strip()
    # 1) Limpiar code fences ```json ... ``` o ``` ... ```
    if "```" in s:
        s = _FENCE_RE.sub("", s).replace("```", "").strip()
    # 2) Intento directo
    try:
        v = json.loads(s)
        return v if isinstance(v, list) else None
    except Exception:
        pass
    # 3) Primer [ ... ] completo via raw_decode (tolerante a lo que venga después)
    decoder = json.JSONDecoder()
    idx = s.find("[")
    while idx != -1:
        try:
            obj, _end = decoder.raw_decode(s, idx)
            if isinstance(obj, list):
                return obj
        except json.JSONDecodeError:
            pass
        idx = s.find("[", idx + 1)
    return None


class LMStudioTranslator:
    """Traduce por lotes vía el servidor OpenAI-compatible de LM Studio.
    El modelo corre en la GPU local; este cliente sólo lanza prompts."""

    def __init__(self, base_url, api_key, model, src, tgt,
                 batch_size=16, max_retries=3, backoff_base=2.0, timeout=120):
        self.base_url   = base_url.rstrip("/")
        self.api_key    = api_key
        self.model      = model
        self.src        = src
        self.tgt        = tgt
        self.batch_size = batch_size
        self.max_retries = max_retries
        self.backoff_base = backoff_base
        self.timeout    = timeout
        self.session    = requests.Session()

    def _build_prompt(self, texts):
        numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(texts))
        return (
            f"Translate the following {len(texts)} short texts from {self.src} to {self.tgt}.\n"
            f"Respond ONLY with a valid JSON array of exactly {len(texts)} strings, "
            f"in the same order, no extra text, no numbering.\n\n"
            f"TEXTS:\n{numbered}\n\nJSON:"
        )

    def _call(self, prompt, n_expected):
        url = f"{self.base_url}/chat/completions"
        headers = {"Authorization": f"Bearer {self.api_key}",
                   "Content-Type": "application/json"}
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
            "stream": False,
        }
        r = self.session.post(url, json=payload, headers=headers, timeout=self.timeout)
        r.raise_for_status()
        content = r.json()["choices"][0]["message"]["content"]
        data = _extract_json_array(content)
        if data is None:
            preview = (content or "")[:200].replace("\n", " ")
            raise ValueError(f"no se encontró JSON array en la respuesta: {preview!r}")
        if not isinstance(data, list) or len(data) != n_expected:
            n = len(data) if isinstance(data, list) else type(data).__name__
            raise ValueError(f"LM Studio devolvió {n} items, esperaba {n_expected}")
        return [str(x).strip() for x in data]

    def translate_batch(self, texts):
        if not texts:
            return []
        prompt = self._build_prompt(texts)
        delay = self.backoff_base
        last_err = None
        for attempt in range(self.max_retries):
            try:
                return self._call(prompt, len(texts))
            except Exception as e:
                last_err = e
                print(f"   ⚠ LM Studio falló (intento {attempt+1}/{self.max_retries}): "
                      f"{type(e).__name__}: {str(e)[:120]} — espero {delay:.0f}s")
                time.sleep(delay)
                delay *= 2
        raise RuntimeError(f"LM Studio agotó reintentos: {last_err}")

    def translate_chunk(self, texts, ids):
        """Traduce una lista paralela (texts, ids) y devuelve records listos para JSONL."""
        now_iso = lambda: datetime.now(timezone.utc).isoformat()
        records = []
        translations = []
        for i in range(0, len(texts), self.batch_size):
            translations.extend(self.translate_batch(texts[i:i+self.batch_size]))
        for did, t in zip(ids, translations):
            records.append({
                "doc_id": did,
                "translation": t,
                "model": self.model,
                "status": "ok",
                "ts": now_iso(),
            })
        return records

print("✅ LMStudioTranslator listo")

In [ ]:
from collections import Counter
import warnings

from tqdm.notebook import tqdm

# Silenciar advertencias de urllib3 que son ruido en consola
warnings.filterwarnings("ignore", category=UserWarning, module="urllib3")

# Cargar progreso existente
done_ok, done_err = load_done_ids(JSONL_PATH)
already = (done_ok | done_err) if not RETRY_ERRORS else done_ok
print(f"📒 JSONL existente: {len(done_ok):,} ok | {len(done_err):,} con error")
if RETRY_ERRORS:
    print("   ⚠ RETRY_ERRORS=True → los 'error' previos se vuelven a intentar.")

# Filtrar trabajo pendiente (NO reprocesa: descarta los doc_id ya en el JSONL)
texts = df[TEXT_COL].astype('string').fillna('').str.strip()
mask = texts.ne('') & ~df[ID_COL].astype(str).isin(already)
work = df.loc[mask, [ID_COL, TEXT_COL]].reset_index(drop=True)
print(f"🛠 Pendientes de traducir: {len(work):,}")

if len(work) == 0:
    print("🎉 Nada que traducir — todo el dataset ya estaba en el JSONL.")
else:
    translator = LMStudioTranslator(
        base_url=LMSTUDIO_BASE_URL,
        api_key=LMSTUDIO_API_KEY,
        model=LMSTUDIO_MODEL,
        src=SRC_LANG,
        tgt=TGT_LANG,
        batch_size=BATCH_SIZE,
        max_retries=MAX_RETRIES,
        backoff_base=BACKOFF_BASE,
    )
    total = len(work)
    t_global = time.time()
    n_chunks = (total + CHUNK_SIZE - 1) // CHUNK_SIZE

    for start in range(0, total, CHUNK_SIZE):
        chunk = work.iloc[start:start+CHUNK_SIZE]
        ids   = chunk[ID_COL].astype(str).tolist()
        txts  = chunk[TEXT_COL].astype(str).tolist()
        chunk_no = start // CHUNK_SIZE + 1
        elapsed = time.time() - t_global
        done = start + len(chunk)
        eta_min = (elapsed / done) * (total - done) / 60
        print(f"\n[Chunk {chunk_no}/{n_chunks}] {len(txts)} textos  |  elapsed {elapsed:.0f}s  |  ETA ~{eta_min:.1f} min")

        # Manejo por batch: si un batch falla tras MAX_RETRIES, se anota como
        # error y se sigue con los demás (no se aborta el chunk entero).
        records = []
        now_iso = lambda: datetime.now(timezone.utc).isoformat()
        n_batches = (len(txts) + BATCH_SIZE - 1) // BATCH_SIZE
        chunk_bar = tqdm(
            range(0, len(txts), BATCH_SIZE),
            total=n_batches,
            desc=f"chunk {chunk_no}/{n_chunks}",
            unit="batch",
            leave=True,
        )
        for i in chunk_bar:
            batch_txts = txts[i:i+BATCH_SIZE]
            batch_ids  = ids[i:i+BATCH_SIZE]
            batch_no   = i // BATCH_SIZE + 1
            try:
                translations = translator.translate_batch(batch_txts)
            except Exception as e:
                # Batch agotó reintentos: registrar como error y seguir.
                # Los textos quedan fuera del JSONL → se reintentan al re-correr.
                print(f"   ✗ Batch {batch_no}/{n_batches} falló: {type(e).__name__}: {str(e)[:120]}")
                chunk_bar.set_postfix(ok=sum(1 for r in records if r['status'] == 'ok'),
                                      err=sum(1 for r in records if r['status'] == 'error'))
                continue
            for did, t in zip(batch_ids, translations):
                records.append({
                    "doc_id": did,
                    "translation": t,
                    "model": LMSTUDIO_MODEL,
                    "status": "ok",
                    "ts": now_iso(),
                })
            chunk_bar.set_postfix(ok=sum(1 for r in records if r['status'] == 'ok'),
                                  err=sum(1 for r in records if r['status'] == 'error'))

        # Persistir INMEDIATAMENTE este chunk (clave para no perder progreso)
        append_translations(records, JSONL_PATH)

        c = Counter((r.get('model'), r.get('status')) for r in records)
        print(f"   modelos/status en chunk: {dict(c)}")

    total_time = time.time() - t_global
    print(f"\n✅ Loop terminado en {total_time/60:.1f} min ({total:,} textos)")

In [ ]:
# Re-leemos TODO el JSONL y mergeamos con el parquet original
records = []
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue

trans_df = pd.DataFrame(records)
# Si un doc_id aparece varias veces (ej. retry), nos quedamos con la última versión
trans_df = trans_df.drop_duplicates(subset=[ID_COL], keep='last')
print(f"Total traducciones únicas: {len(trans_df):,}")

cols_keep = [c for c in (ID_COL, 'translation', 'model', 'status', 'error', 'ts') if c in trans_df.columns]
merged = df.merge(trans_df[cols_keep], on=ID_COL, how='left')

merged.to_parquet(PARQUET_OUT, index=False)
print(f"\nGuardado: {PARQUET_OUT}  ({len(merged):,} filas)")

print("\nDistribución de modelos:")
print(merged['model'].fillna('NOT_TRANSLATED').value_counts())

print("\nDistribución de status:")
print(merged['status'].fillna('PENDING').value_counts())
